In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    trim,
    lower,
    regexp_replace,
    length,
    count
)

spark = (
    SparkSession.builder
    .appName("Gaming Analytics - Data Cleaning")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config( "spark.local.dir", "D:/spark_temp")
    .config( "spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [37]:
print("\n========== LOAD RAW DATA ==========\n")
games = spark.read.csv(
    "../data/bronze/steam_games.csv",
    header=True,
    inferSchema=True
)
reviews = spark.read.csv(
    "../data/bronze/steam_reviews.csv",
    header=True,
    inferSchema=True
)
print("DONE")


========== LOAD RAW DATA ==========

DONE


In [38]:
print("\n========== REMOVE DUPLICATES ==========\n")
games = games.dropDuplicates(["appid"])
games = games.dropna(
    subset=[
        "appid",
        "name"
    ]
)
reviews = reviews.dropDuplicates(
    [
        "app_id",
        "review_text"
    ]
)
print("DONE")


========== REMOVE DUPLICATES ==========

DONE


In [39]:
print("\n========== CLEAN TEXT COLUMNS ==========\n")
text_columns = [
    "name",
    "developer",
    "publisher",
    "genres"
]
for c in text_columns:
    games = games.withColumn(
        c,
        trim(col(c))
    )

print("DONE")


========== CLEAN TEXT COLUMNS ==========

DONE


In [40]:
print("\n========== CONVERT NUMERIC VALUES ==========\n")
games = games.withColumn(
    "positive_ratings",
    col("positive_ratings").cast("integer")
)
games = games.withColumn(
    "price",
    regexp_replace(
        col("price"),
        "[^0-9.]",
        ""
    )
)
games = games.withColumn(
    "price",
    col("price").cast("double")
)
reviews = reviews.withColumn(
    "review_score",
    col("review_score").cast("integer")
)
print("DONE")


========== CONVERT NUMERIC VALUES ==========

DONE


In [41]:
print("\n========== REMOVE EMPTY REVIEWS ==========\n")
reviews = reviews.dropna(
    subset=[
        "app_id",
        "review_text"
    ]
)
print("DONE")

print("\n========== CONVERT TO LOWERCASE ==========\n")
reviews = reviews.withColumn(
    "review_text",
    lower(col("review_text"))
)
print("DONE")

print("\n========== REMOVE UNNECESSARY CHARACTERS/SYMBOLS ==========\n")
reviews = reviews.withColumn(
    "review_text",
    regexp_replace(
        col("review_text"),
        "[^a-zA-Z0-9 ]",
        ""
    )
)
print("DONE")

print("\n========== REMOVE EMPTY CLEANED REVIEWS ==========\n")
reviews = reviews.filter(
    length(col("review_text")) > 0
)
print("DONE")

print("\n========== REMOVE POSSIBLE DUPLICATE COLUMN ==========\n")
reviews = reviews.drop("app_name")
print("DONE")

print("\n========== FILTER REVIEWS BY LENGTH ==========\n")
reviews = reviews.filter(
    length(col("review_text")) <= 5000
)
print("DONE")

print("\n========== REMOVE INVALID REVIEW SCORES ==========\n")
reviews = reviews.dropna(
    subset=[
        "review_score"
    ]
)
reviews = reviews.fillna(
    {
        "review_votes": 0
    }
)
print("DONE")


========== REMOVE EMPTY REVIEWS ==========

DONE

========== CONVERT TO LOWERCASE ==========

DONE

========== REMOVE UNNECESSARY CHARACTERS/SYMBOLS ==========

DONE

========== REMOVE EMPTY CLEANED REVIEWS ==========

DONE

========== REMOVE POSSIBLE DUPLICATE COLUMN ==========

DONE

========== FILTER REVIEWS BY LENGTH ==========

DONE

========== REMOVE INVALID REVIEW SCORES ==========

DONE


In [42]:
print("\n========== SAVE SILVER DATA ==========\n")

games.write \
    .mode("overwrite") \
    .parquet(
        "../data/silver/games_clean"
    )


reviews.write \
    .mode("overwrite") \
    .parquet(
        "../data/silver/reviews_clean"
    )


print("Silver layer saved successfully")


========== SAVE SILVER DATA ==========

Silver layer saved successfully


In [43]:
print("\n========== PREVIEW GAMES ==========\n")

games.show(5, truncate=False)

print("\n========== PREVIEW REVIEWS ==========\n")

reviews.show(5, truncate=False)


========== PREVIEW GAMES ==========

+-----+-----------------------+------------+-------+---------+---------+-----------------+------------+---------------------------------------------------------------------------------------------------------------------------------+------+----------------------------+------------+----------------+----------------+----------------+---------------+-----------------+-----+
|appid|name                   |release_date|english|developer|publisher|platforms        |required_age|categories                                                                                                                       |genres|steamspy_tags               |achievements|positive_ratings|negative_ratings|average_playtime|median_playtime|owners           |price|
+-----+-----------------------+------------+-------+---------+---------+-----------------+------------+---------------------------------------------------------------------------------------------------------------

In [44]:
print("\n========== SILVER DATA QUALITY CHECK ==========\n")

print("Games rows:")
print(games.count())

print("\nReviews rows:")
print(reviews.count())

print("\nEmpty review text:")
reviews.filter(
    length(col("review_text")) == 0
).count()

print("\nNull values:")
reviews.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in reviews.columns
]).show()


========== SILVER DATA QUALITY CHECK ==========

Games rows:


27075

Reviews rows:
4584959

Empty review text:

Null values:
+------+-----------+------------+------------+
|app_id|review_text|review_score|review_votes|
+------+-----------+------------+------------+
|     0|          0|           0|           0|
+------+-----------+------------+------------+

